In [2]:
import pandas as pd
import numpy as np
import torch 
import torch.nn as nn

In [3]:
from pathlib import Path
from scipy.sparse import csr_matrix, hstack
from sklearn.decomposition import TruncatedSVD
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import roc_auc_score

In [4]:
data_dir = Path("../data/raw")

In [5]:
documents_categories = pd.read_csv(data_dir / "documents_categories.csv.zip")
documents_topics = pd.read_csv(data_dir / "documents_topics.csv.zip")
documents_entities = pd.read_csv(data_dir / "documents_entities.csv.zip")

In [6]:
# document id 매핑 
all_doc_ids = pd.Index(sorted(
    set(documents_categories["document_id"])
    | set(documents_topics["document_id"])
    | set(documents_entities["document_id"])
))
doc_id_to_idx = {doc_id: i for i, doc_id in enumerate(all_doc_ids)}


In [7]:
category_ids = sorted(documents_categories["category_id"].unique())
category_id_to_idx = {v: i for i, v in enumerate(category_ids)}

In [8]:
topic_ids = sorted(documents_topics["topic_id"].unique())
topic_id_to_idx = {v: i for i, v in enumerate(topic_ids)}

In [9]:
entity_ids = sorted(documents_entities["entity_id"].unique())
entity_id_to_idx = {v: i for i, v in enumerate(entity_ids)}

In [10]:
cat_rows = documents_categories["document_id"].map(doc_id_to_idx)
cat_cols = documents_categories["category_id"].map(category_id_to_idx)
category_matrix = csr_matrix(
    (documents_categories["confidence_level"], (cat_rows, cat_cols)),
    shape=(len(all_doc_ids), len(category_ids)),
)

topic_rows = documents_topics["document_id"].map(doc_id_to_idx)
topic_cols = documents_topics["topic_id"].map(topic_id_to_idx)
topic_matrix = csr_matrix(
    (documents_topics["confidence_level"], (topic_rows, topic_cols)),
    shape=(len(all_doc_ids), len(topic_ids)),
)

entity_rows = documents_entities["document_id"].map(doc_id_to_idx)
entity_cols = documents_entities["entity_id"].map(entity_id_to_idx)
entity_matrix = csr_matrix(
    (documents_entities["confidence_level"], (entity_rows, entity_cols)),
    shape=(len(all_doc_ids), len(entity_ids)),
)

content_features_sparse = hstack([category_matrix, topic_matrix, entity_matrix]).tocsr()
content_features_sparse.shape

(2919892, 1326406)

In [11]:
# 128차원으로 압축 
svd = TruncatedSVD(n_components=128, random_state=42)
content_embeddings = svd.fit_transform(content_features_sparse)
content_embeddings.shape

(2919892, 128)

In [12]:
# 저차원 임베딩 조회 함수 
def get_content_vector(doc_id):
    idx = doc_id_to_idx.get(doc_id)
    if idx is None:
        return np.zeros(content_embeddings.shape[1], dtype=np.float32)
    return content_embeddings[idx].astype(np.float32)

In [13]:
# 클릭 데이터 샘플링과 조인
clicks_train = pd.read_csv("../data/raw/clicks_train.csv.zip")
events = pd.read_csv("../data/raw/events.csv.zip", dtype={"platform": str})
promoted_content = pd.read_csv("../data/raw/promoted_content.csv.zip")

sample_display_ids = clicks_train["display_id"].drop_duplicates().sample(1_000_000, random_state=42)
clicks_sample = clicks_train[clicks_train["display_id"].isin(sample_display_ids)]

merged = clicks_sample.merge(events, on="display_id").merge(
    promoted_content, on="ad_id", suffixes=("_view", "_ad")
)

val_display_ids = merged["display_id"].drop_duplicates().sample(frac=0.2, random_state=42)
val = merged[merged["display_id"].isin(val_display_ids)]
train = merged[~merged["display_id"].isin(val_display_ids)]

In [26]:
def build_session_groups(df):
    return df.groupby("display_id").agg({"document_id_ad": list, "clicked": list})

train_sessions = build_session_groups(train)
val_sessions = build_session_groups(val)

In [29]:
MAX_CANDIDATES = 12

class SessionDataset(Dataset):
    def __init__(self, sessions_df, get_content_vector, embed_dim):
        self.doc_lists = sessions_df["document_id_ad"].tolist()
        self.click_lists = sessions_df["clicked"].tolist()
        self.get_content_vector = get_content_vector
        self.embed_dim = embed_dim

    def __len__(self):
        return len(self.doc_lists)

    def __getitem__(self, idx):
        docs = self.doc_lists[idx][:MAX_CANDIDATES]
        clicks = self.click_lists[idx][:MAX_CANDIDATES]

        vectors = np.stack([self.get_content_vector(d) for d in docs])
        clicked_idx = clicks.index(1)

        return (
            torch.tensor(vectors, dtype=torch.float32),
            torch.tensor(clicked_idx, dtype=torch.long),
            len(docs),
        )

In [30]:
def collate_sessions(batch):
    vectors, clicked_idx, lengths = zip(*batch)
    lengths = torch.tensor(lengths)
    max_len = lengths.max().item()

    padded = torch.zeros(len(batch), max_len, vectors[0].shape[1])
    mask = torch.zeros(len(batch), max_len, dtype=torch.bool)
    for i, v in enumerate(vectors):
        padded[i, :v.shape[0]] = v
        mask[i, :v.shape[0]] = True

    clicked_idx = torch.stack(clicked_idx)
    return padded, mask, clicked_idx

In [31]:
class SessionAttentionModel(nn.Module):
    def __init__(self, input_dim, embed_dim=32, n_heads=4):
        super().__init__()
        self.encode = nn.Sequential(nn.Linear(input_dim, 64), nn.ReLU(), nn.Linear(64, embed_dim))
        self.attn = nn.MultiheadAttention(embed_dim, n_heads, batch_first=True)
        self.score = nn.Linear(embed_dim, 1)

    def forward(self, padded, mask):
        x = self.encode(padded)
        key_padding_mask = ~mask
        attn_out, _ = self.attn(x, x, x, key_padding_mask=key_padding_mask)
        scores = self.score(attn_out).squeeze(-1)
        scores = scores.masked_fill(~mask, float("-inf"))
        return scores

In [32]:
train_dataset = SessionDataset(train_sessions, get_content_vector, embed_dim=content_embeddings.shape[1])
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, collate_fn=collate_sessions)

model = SessionAttentionModel(input_dim=content_embeddings.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    model.train()
    epoch_loss = 0
    for padded, mask, clicked_idx in train_loader:
        optimizer.zero_grad()
        scores = model(padded, mask)
        loss = nn.functional.cross_entropy(scores, clicked_idx)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(epoch, epoch_loss / len(train_loader))

0 1.504958540878296
1 1.4725065422821044
2 1.4617999271011353
3 1.4560216513442994
4 1.4513021897125244


In [33]:
val_dataset = SessionDataset(val_sessions, get_content_vector, embed_dim=content_embeddings.shape[1])
val_loader = DataLoader(val_dataset, batch_size=256, collate_fn=collate_sessions)

model.eval()
all_scores, all_labels = [], []
with torch.no_grad():
    for padded, mask, clicked_idx in val_loader:
        scores = model(padded, mask)
        probs = torch.softmax(scores, dim=1)
        for i in range(len(clicked_idx)):
            n = mask[i].sum().item()
            labels_i = torch.zeros(n)
            labels_i[clicked_idx[i]] = 1
            all_scores.extend(probs[i, :n].tolist())
            all_labels.extend(labels_i.tolist())

roc_auc_score(all_labels, all_scores)

0.7107003384639783

In [37]:
display_ids = val_sessions.index.tolist()
doc_lists = val_sessions["document_id_ad"].tolist()

records = []
model.eval()
ptr = 0
with torch.no_grad():
    for padded, mask, clicked_idx in val_loader:
        batch_size = padded.shape[0]
        scores = model(padded, mask)
        probs = torch.softmax(scores, dim=1)
        for i in range(batch_size):
            n = mask[i].sum().item()
            did = display_ids[ptr]
            docs = doc_lists[ptr][:n]
            for j in range(n):
                records.append({
                    "display_id": did,
                    "document_id_ad": docs[j],
                    "score_session": probs[i, j].item(),
                    "clicked": 1 if j == clicked_idx[i].item() else 0,
                })
            ptr += 1

In [38]:
val_with_scores = pd.DataFrame(records)
val_with_scores.to_csv("../data/processed/val_with_scores.csv", index=False)

import pickle
with open("../data/processed/content_embeddings.pkl", "wb") as f:
    pickle.dump({"doc_id_to_idx": doc_id_to_idx, "embeddings": content_embeddings}, f)